In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

### Parallel Chains

In [12]:
prompt_main = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Give a 3 line summary of the topic: {topic}")
])

llm = ChatOpenAI(model="gpt-5-mini", temperature=0.2)

parser = StrOutputParser()

In [9]:
def dict_maker(text: str) -> dict:
    return {"text": text}

dict_maker_runnable = RunnableLambda(dict_maker)

chain1 = prompt_main | llm | parser | dict_maker_runnable

In [10]:
def pros(d: dict):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "Fetch the topic from the following text: {text}. Give 3 main pros of the topic in bullet points.")
    ])
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
    parser = StrOutputParser()
    chain = prompt | llm | parser
    return chain.invoke({"text": d["text"]})

def cons(d: dict):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "Fetch the topic from the following text: {text}. Give 3 main cons of the topic in bullet points.")
    ])
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
    parser = StrOutputParser()
    chain = prompt | llm | parser
    return chain.invoke({"text": d["text"]})

pros_runnable = RunnableLambda(pros)
cons_runnable = RunnableLambda(cons)

In [8]:
def merge_output(d):
    return {
        "Pros": d["pros"],
        "Cons": d["cons"]
    }

merge_runnable = RunnableLambda(merge_output)

In [13]:
final_chain = (chain1 |
            RunnableParallel(pros=pros_runnable, cons=cons_runnable) |
            merge_runnable
)

response = final_chain.invoke({"topic": "smartphones"})
print(response)

{'Pros': '**Topic:** Smartphones\n\n**Main Pros:**\n\n*   Facilitate communication\n*   Provide extensive information access\n*   Offer versatility through a vast ecosystem of applications and multimedia features', 'Cons': '**Topic:** Smartphones\n\n**3 Main Cons:**\n\n*   **Potential for addiction and distraction:** Leading to reduced productivity, impaired focus, and decreased face-to-face social interaction.\n*   **Privacy concerns and security risks:** Smartphones collect vast amounts of personal data, making users vulnerable to data breaches, tracking, and identity theft.\n*   **Negative impacts on physical and mental health:** Prolonged use can cause eye strain, sleep disruption (due to blue light), "tech neck," and contribute to anxiety, depression, or comparison culture (especially with social media).'}


In [14]:
print(response["Pros"])
print(response["Cons"])

**Topic:** Smartphones

**Main Pros:**

*   Facilitate communication
*   Provide extensive information access
*   Offer versatility through a vast ecosystem of applications and multimedia features
**Topic:** Smartphones

**3 Main Cons:**

*   **Potential for addiction and distraction:** Leading to reduced productivity, impaired focus, and decreased face-to-face social interaction.
*   **Privacy concerns and security risks:** Smartphones collect vast amounts of personal data, making users vulnerable to data breaches, tracking, and identity theft.
*   **Negative impacts on physical and mental health:** Prolonged use can cause eye strain, sleep disruption (due to blue light), "tech neck," and contribute to anxiety, depression, or comparison culture (especially with social media).
